In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[
        # redact : 개인정보를 완전히 가림
        PIIMiddleware("email", strategy="redact", apply_to_input=True)
    ],
)

In [4]:
prompt = "안녕하세요. 제 이메일은 kim@example.com 입니다."

In [5]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

response

{'messages': [HumanMessage(content='안녕하세요. 제 이메일은 [REDACTED_EMAIL] 입니다.', additional_kwargs={}, response_metadata={}, id='5254a8e7-f083-4c9b-8782-c072717ed6d5'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요! 정보를 공유해 주셔서 감사합니다. \n\n개인정보 보호를 위해, 이렇게 공개적인 채팅창에는 이메일 주소나 전화번호 같은 민감한 개인정보를 직접 입력하지 않는 것이 좋습니다. \n\n혹시 제가 이메일과 관련하여 도와드릴 일이 있으신가요? 필요하신 작업이 있다면 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHmESySNuKqSmlCiE1fIjM2/nVjs+PmHMLP1RF5B4xIwgwz9RIk8LwGT+B/qRD'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e10dd-085a-7603-8ad6-c858943e7f3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 74, 'total_tokens': 91, 'input_token_details': {'cache_read': 0}})]}

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[
        # mask : 마지막 일부만 남기고 마스킹
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)

In [18]:
prompt = "안녕하세요. 제 카드번호는 1234123443214321 입니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

response

{'messages': [HumanMessage(content='안녕하세요. 제 카드번호는 ************4321 입니다.', additional_kwargs={}, response_metadata={}, id='59dffdca-b547-47f3-89f6-71e3e438f11a'),
  AIMessage(content=[{'type': 'text', 'text': '개인정보 보호를 위해 **카드번호 전체나 일부를 온라인상에 올리는 것은 매우 위험합니다.**\n\n지금 즉시 해당 메시지를 삭제하시고, 아래의 주의사항을 확인해 주세요.\n\n1.  **정보 삭제:** 이미 입력하신 메시지를 삭제해 주세요.\n2.  **카드 분실신고/재발급:** 만약 이 정보가 타인에게 노출되었을 가능성이 있다면, 즉시 해당 카드사 고객센터에 연락하여 **카드 정지 및 재발급**을 받으시는 것을 권장합니다.\n3.  **보안 주의:** 어떠한 경우에도 카드 번호, 유효기간, CVC 번호, 비밀번호 등을 공개된 채팅창이나 신뢰할 수 없는 곳에 입력하지 마세요.\n\n저는 AI 모델로서 사용자의 결제 정보를 저장하거나 조회할 수 없으며, 이러한 개인정보는 입력하지 않도록 주의해 주시기 바랍니다. 도움이 필요하신 다른 내용이 있으신가요?', 'extras': {'signature': 'EjQKMgEMOdbHfEKC5MHQOdsvOnIXuuI+45LcE8Qoe1SBOOugfALufUK9HdRhkgRk7W5CXK/V'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e10ed-311d-7e93-869c-37a91d359bb4-0', tool_calls=[], invalid_tool_calls=[], usage_m

In [ ]:
phone_number_regex = r'01[016789]-?\d{3,4}-?\d{4}'

phone_middleware = PIIMiddleware(
    pii_type="phone_number",
    detector=phone_number_regex,
    # strategy="mask",
    # strategy="block", # 민감 정보가 감지되는 즉시 에이전트 실행을 중단 -> 오류 발생
    strategy="hash", # 민감 정보를 일관된 해시값으로 변환
    apply_to_input=True
)

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[phone_middleware],
)

In [27]:
prompt = "안녕하세요. 제 폰 번호는 010-1234-5678 입니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

response

{'messages': [HumanMessage(content='안녕하세요. 제 폰 번호는 <phone_number_hash:e07c05a8> 입니다.', additional_kwargs={}, response_metadata={}, id='6b9f8fcc-eb3b-45f6-88bb-cef6f1d90980'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요! 말씀해주신 전화번호 정보는 보안을 위해 시스템상에서 암호화된 값으로 인식되었습니다.\n\n혹시 도움이 필요하시거나 궁금한 점이 있으신가요? 무엇을 도와드릴까요?', 'extras': {'signature': 'EjQKMgEMOdbHE8QJt+AzLM+JzaBmCWz0HEepRoL2nVmH6cKmtd9RxywpvD/lnrUjM3YnScgb'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e10f4-c972-7d81-9f36-8913ec5c4635-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 46, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})]}